In [1]:
import sys

import pm4py

import pandas as pd
import numpy as np

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts
from model.next_event_model import ProcessLSTM, train_ProcessLSTM, validate_ProcessLSTM

### --- Preprocess dataset ---

In [2]:
set_seed(seed=42)

In [3]:
log = pm4py.read_xes("../../data/bpic19.xes")

C:\Users\dcoralage\Downloads\counterfactual_exp\counterfactual_env\lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
C:\Users\dcoralage\Downloads\counterfactual_exp\counterfactual_env\lib\site-packages\pm4py\util\dt_parsing\parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/251734 [00:00<?, ?it/s]

In [4]:
df = pm4py.convert_to_dataframe(log)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1595923 entries, 0 to 1595922
Data columns (total 21 columns):
 #   Column                          Non-Null Count    Dtype              
---  ------                          --------------    -----              
 0   User                            1595923 non-null  object             
 1   org:resource                    1595923 non-null  object             
 2   concept:name                    1595923 non-null  object             
 3   Cumulative net worth (EUR)      1595923 non-null  float64            
 4   time:timestamp                  1595923 non-null  datetime64[ns, UTC]
 5   case:Spend area text            1595923 non-null  object             
 6   case:Company                    1595923 non-null  object             
 7   case:Document Type              1595923 non-null  object             
 8   case:Sub spend area text        1595923 non-null  object             
 9   case:Purchasing Document        1595923 non-null  object 

In [6]:
df.isnull().any()

User                              False
org:resource                      False
concept:name                      False
Cumulative net worth (EUR)        False
time:timestamp                    False
case:Spend area text              False
case:Company                      False
case:Document Type                False
case:Sub spend area text          False
case:Purchasing Document          False
case:Purch. Doc. Category name    False
case:Vendor                       False
case:Item Type                    False
case:Item Category                False
case:Spend classification text    False
case:Source                       False
case:Name                         False
case:GR-Based Inv. Verif.         False
case:Item                         False
case:concept:name                 False
case:Goods Receipt                False
dtype: bool

In [7]:
df = df.drop(columns=['case:Purchasing Document', 'case:Item', 'case:Name', 'User', 'org:resource', 'case:Company', 'case:Vendor'])

In [8]:
df['case:concept:name'] = df['case:concept:name'].astype('string')
df['concept:name'] = df['concept:name'].astype('string')

df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], errors='coerce')

df['case:Spend area text'] = df['case:Spend area text'].astype('string')
df['case:Document Type'] = df['case:Document Type'].astype('string')
df['case:Sub spend area text'] = df['case:Sub spend area text'].astype('string')
df['case:Purch. Doc. Category name'] = df['case:Purch. Doc. Category name'].astype('string')
df['case:Item Type'] = df['case:Item Type'].astype('string')
df['case:Item Category'] = df['case:Item Category'].astype('string')
df['case:Spend classification text'] = df['case:Spend classification text'].astype('string')
df['case:Source'] = df['case:Source'].astype('string')
df['case:GR-Based Inv. Verif.'] = df['case:GR-Based Inv. Verif.'].astype('string')
df['case:Goods Receipt'] = df['case:Goods Receipt'].astype('string')

df['Cumulative net worth (EUR)'] = df['Cumulative net worth (EUR)'].astype(np.float32)

In [9]:
df = df.sort_values(by=['case:concept:name', 'time:timestamp'], ascending=[True, True])

In [10]:
df['time_delta'] = df.groupby('case:concept:name')['time:timestamp'].diff()
df['time_delta'] = df['time_delta'].dt.total_seconds()
df['time_delta'] = df['time_delta'].fillna(0)

In [11]:
exclude_cols = ["case:concept:name", "time:timestamp"]

sorted_cols = sorted(
    [c for c in df.columns if c not in exclude_cols]
)

df = df[exclude_cols + sorted_cols]

In [12]:
df.head(20)

,case:concept:name,time:timestamp,Cumulative net worth (EUR),case:Document Type,case:GR-Based Inv. Verif.,case:Goods Receipt,case:Item Category,case:Item Type,case:Purch. Doc. Category name,case:Source,case:Spend area text,case:Spend classification text,case:Sub spend area text,concept:name,time_delta
0,2000000000_00001,2018-01-02 12:53:00+00:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Created,0.0
1,2000000000_00001,2018-01-02 13:53:00+00:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Complete,3600.0
2,2000000000_00001,2018-01-02 13:53:00+00:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Awaiting Approval,0.0
3,2000000000_00001,2018-01-02 13:53:00+00:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Document Completed,0.0
4,2000000000_00001,2018-01-02 13:53:00+00:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: In Transfer to Execution Syst.,0.0
5,2000000000_00001,2018-01-02 13:53:00+00:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Ordered,0.0
6,2000000000_00001,2018-01-02 13:53:00+00:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Change was Transmitted,0.0
7,2000000000_00001,2018-01-02 13:53:00+00:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Create Purchase Order Item,0.0
8,2000000000_00001,2018-01-02 22:59:00+00:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Vendor creates invoice,32760.0
9,2000000000_00001,2018-03-06 06:44:00+00:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Record Goods Receipt,5384700.0


In [13]:
num_cases = df['case:concept:name'].nunique()
print(f"Total number of unique cases: {num_cases}")

Total number of unique cases: 251734


### --- Feature Configurations ---

In [14]:
# --- Define feature specs ---
feature_specs = {

    "case:Document Type": {
        "type":            "categorical",
        "level":           "case",
        "vary":            True,
    },

    "case:GR-Based Inv. Verif.": {
        "type":            "categorical",
        "level":           "case",
        "vary":            True,
    },

    "case:Goods Receipt": {
        "type":            "categorical",
        "level":           "case",
        "vary":            True,
    },

    "case:Item Category": {
        "type":            "categorical",
        "level":           "case",
        "vary":            True,
    },

    "case:Item Type": {
        "type":            "categorical",
        "level":           "case",
        "vary":            True,
    },

    "case:Purch. Doc. Category name": {
        "type":            "categorical",
        "level":           "case",
        "vary":            True,
    },

    "case:Source": {
        "type":            "categorical",
        "level":           "case",
        "vary":            True,
    },

    "case:Spend area text": {
        "type":            "categorical",
        "level":           "case",
        "vary":            True,
    },

    "case:Spend classification text": {
        "type":            "categorical",
        "level":           "case",
        "vary":            True,
    },

    "case:Sub spend area text": {
        "type":            "categorical",
        "level":           "case",
        "vary":            True,
    },

    "time_delta": {
        "type":            "continuous",
        "level":           "event",
        "vary":            True,
        "quantile_low":    0.20,
        "quantile_high":   0.80, 
    },

    "Cumulative net worth (EUR)": {
        "type":            "continuous",
        "level":           "event",
        "vary":            True,
        "quantile_low":    0.05,
        "quantile_high":   0.90, 
    },

    # immutable
    "concept:name": {
        "type":            "categorical", 
        "level":           "event",
        "vary":            False
    },
}

In [15]:
feature_config = FeatureConfig.from_dataframe(
    df=df,
    feature_specs=feature_specs,
    activity_feature="concept:name",
    is_robust=True,
    default_quantile_low=0.05,
    default_quantile_high=0.95
)

feature_config.save()

In [16]:
# feature_config = FeatureConfig.load()

In [17]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['Cumulative net worth (EUR)', 'case:Document Type', 'case:GR-Based Inv. Verif.', 'case:Goods Receipt', 'case:Item Category', 'case:Item Type', 'case:Purch. Doc. Category name', 'case:Source', 'case:Spend area text', 'case:Spend classification text', 'case:Sub spend area text', 'concept:name', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
case:Document Type             categorical    case     yes    ['EC Purchase order', 'Framework order', 'Standard PO'] N/A        data_derived        
case:GR-Based Inv. Verif.      categorical    case     yes    ['False', 'True

### --- Next event prediction model ---

In [18]:
cat_cols = df.select_dtypes(include=["string"]).columns
for col in cat_cols:
    df[col] = df[col].fillna("NA").astype('string')

In [19]:
case_ids = df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = df[df["case:concept:name"].isin(train_cases)].copy()
val_df   = df[df["case:concept:name"].isin(val_cases)].copy()

In [20]:
preprocessor_artifacts = PreprocessorArtifacts.build(
    df=train_df,
    feature_config=feature_config,      
    scaler_type="robust",
)

preprocessor_artifacts.save()

In [21]:
# preprocessor_artifacts = PreprocessorArtifacts.load()

In [22]:
preprocessor_artifacts.summary()

===================PreprocessorArtifacts====================
  scaler:              RobustScaler
  encoders:            ['case:Document Type', 'case:GR-Based Inv. Verif.', 'case:Goods Receipt', 'case:Item Category', 'case:Item Type', 'case:Purch. Doc. Category name', 'case:Source', 'case:Spend area text', 'case:Spend classification text', 'case:Sub spend area text', 'concept:name']
  activity_prototypes: 42 activities


In [23]:
# Transform nan cols to 0
float_cols = df.select_dtypes(include=["float32", "float64"]).columns
df[float_cols] = df[float_cols].fillna(0)
train_df[float_cols] = train_df[float_cols].fillna(0)
val_df[float_cols] = val_df[float_cols].fillna(0)

In [24]:
train_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    case_id_field="case:concept:name", 
    df=train_df,
    sort_field="time:timestamp"
)

val_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    case_id_field="case:concept:name", 
    df=val_df,
    sort_field="time:timestamp"
)

In [25]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [26]:
print(preprocessor_artifacts.get_categorical_feature_cardinality())

{'dynamic_categorical_info': {'concept:name': 42}, 'static_categorical_info': {'case:Document Type': 3, 'case:GR-Based Inv. Verif.': 2, 'case:Goods Receipt': 2, 'case:Item Category': 4, 'case:Item Type': 6, 'case:Purch. Doc. Category name': 1, 'case:Source': 1, 'case:Spend area text': 21, 'case:Spend classification text': 4, 'case:Sub spend area text': 136}}


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [28]:
device

device(type='cuda')

In [29]:
criterion = torch.nn.CrossEntropyLoss()

In [30]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic19-model_output.txt")

Epoch 020/100 | Train Loss: 0.5373 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.5183 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.5021 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.4849 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.4751 | LR: 1.00e-06
Time taken for next event model (training): 28089.514009 seconds
Time taken for next event model (validation): 19.124914 seconds
Val loss: {'loss': 0.5659209988968904, 'accuracy': 0.8153765182790722, 'f1_macro': 0.5446889249103007, 'f1_weighted': 0.8039740693810528}


In [31]:
embedding_metadata = preprocessor_artifacts.get_embedding_metadata()

model = ProcessLSTM(
    dynamic_categorical_info=embedding_metadata["dynamic_categorical_info"],
    static_categorical_info=embedding_metadata["static_categorical_info"],
    n_dynamic_continuous=embedding_metadata["n_dynamic_continuous"],
    n_static_continuous=embedding_metadata["n_static_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ProcessLSTM(
    model=model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

model.save()

In [32]:
# model = ProcessLSTM.load()

In [33]:
val_loss = validate_ProcessLSTM(
    model=model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [34]:
import math

if df["time:timestamp"].dt.tz is not None:
    df["time:timestamp"] = df["time:timestamp"].dt.tz_convert(None)

max_rows = 1_048_575

with pd.ExcelWriter("../../data/bpic19.xlsx", engine="openpyxl") as writer:
    for i in range(math.ceil(len(df) / max_rows)):
        start = i * max_rows
        end = min((i + 1) * max_rows, len(df))
        df.iloc[start:end].to_excel(
            writer,
            sheet_name=f"Part_{i+1}",
            index=False,
        )

In [35]:
sys.stdout = original_stdout
log_file.close()